# Этап 3: Анализ документа и поиск логических проблем (FRIDA)

Этот ноутбук реализует основную цель проекта:
- анализ семантической структуры курсовой,
- выявление вероятных логических проблем,
- формирование интерпретируемого отчета.

В ноутбуке используются эмбеддинги выбранной модели `FRIDA`.


## 1) Импорты и конфигурация

Что делает ячейка:
- импортирует нужные библиотеки,
- задает пути к данным,
- задает параметры и пороги.


In [1]:
import json
from pathlib import Path

import numpy as np


BASE = Path("out")
META_PATH = BASE / "meta.json"
EMB_PATH = BASE / "emb_FRIDA.npy"
REPORT_PATH = BASE / "document_metrics_report.json"

# Сколько самых слабых переходов включать в отчет.
TOP_N_TRANSITIONS = 5

# Порог избыточности для близостей main-main.
REDUNDANCY_THRESHOLD = 0.90

# Порог Z-score для выбросов среди соседних пар (нижний хвост).
ADJ_OUTLIER_Z = -1.0

print("meta:", META_PATH)
print("emb:", EMB_PATH)
print("report:", REPORT_PATH)


meta: out\meta.json
emb: out\emb_FRIDA.npy
report: out\document_metrics_report.json


## 2) Загрузка данных

Что делает ячейка:
- загружает метаданные сегментов,
- загружает эмбеддинги FRIDA,
- проверяет согласованность размеров.


In [2]:
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

emb = np.load(EMB_PATH)

n_segments = len(meta)
if emb.shape[0] != n_segments:
    raise ValueError(f"Mismatch: embeddings={emb.shape[0]} vs meta={n_segments}")

print("segments:", n_segments)
print("emb shape:", emb.shape)


segments: 6
emb shape: (6, 1536)


## 3) Матрица сходства и соседняя близость

Что делает ячейка:
- строит полную матрицу cosine similarity `S`,
- считает близость соседних сегментов `adj[i] = S[i, i+1]`.


In [3]:
def cosine_matrix(vectors: np.ndarray) -> np.ndarray:
    vectors = vectors.astype(np.float32)
    vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
    return vectors @ vectors.T


S = cosine_matrix(emb)
adj = np.array([float(S[i, i + 1]) for i in range(n_segments - 1)], dtype=np.float32)

print("S shape:", S.shape)
print("adj size:", adj.size)
print("adj mean:", round(float(adj.mean()) if adj.size else float("nan"), 4))


S shape: (6, 6)
adj size: 5
adj mean: 0.6693


## 4) Индексы ролей и структуры

Что делает ячейка:
- выделяет индексы ролей (`intro`, `main`, `conclusion`),
- собирает группы по родительским разделам для диагностики.


In [4]:
roles = np.array([m.get("role", "") for m in meta])
parents = np.array([str(m.get("parent", "")) for m in meta])
paths = [str(m.get("path", "")) for m in meta]

main_idx = np.where(roles == "main")[0]
intro_idx = np.where(roles == "intro")[0]
conc_idx = np.where(roles == "conclusion")[0]

intro_i = int(intro_idx[0]) if intro_idx.size else None
conc_i = int(conc_idx[0]) if conc_idx.size else None

parent_to_indices = {}
for i, p in enumerate(parents):
    parent_to_indices.setdefault(p, []).append(i)

print("main:", len(main_idx), "intro:", intro_i, "conclusion:", conc_i)
print("parent groups:", len(parent_to_indices))


main: 4 intro: 0 conclusion: 5
parent groups: 6


## 5) Локальная связность и проблемные переходы

Что делает ячейка:
- считает статистики локальной связности по `adj`,
- находит слабые переходы (кандидаты на логические разрывы),
- собирает top-N самых слабых переходов.


In [5]:
adj_mean = float(adj.mean()) if adj.size else float("nan")
adj_std = float(adj.std(ddof=0)) if adj.size else float("nan")

if np.isfinite(adj_std) and adj_std > 0:
    adj_z = (adj - adj_mean) / adj_std
else:
    adj_z = np.zeros_like(adj)

weak_transition_indices = [int(i) for i, z in enumerate(adj_z) if float(z) <= ADJ_OUTLIER_Z]

# Дополнительно всегда сохраняем top-N самых слабых переходов для интерпретируемости.
if adj.size:
    weakest_order = np.argsort(adj)[: min(TOP_N_TRANSITIONS, adj.size)]
else:
    weakest_order = np.array([], dtype=np.int64)

weakest_transitions = []
for i in weakest_order:
    i = int(i)
    weakest_transitions.append(
        {
            "from_index": i,
            "to_index": i + 1,
            "from_path": paths[i],
            "to_path": paths[i + 1],
            "adj_similarity": float(adj[i]),
            "adj_z": float(adj_z[i]),
        }
    )

print("adj mean/std:", round(adj_mean, 4), "/", round(adj_std, 4))
print("weak transitions by z-threshold:", len(weak_transition_indices))
print("top weakest transitions:", len(weakest_transitions))


adj mean/std: 0.6693 / 0.1162
weak transitions by z-threshold: 1
top weakest transitions: 5


## 6) Согласованность введения/основной части/заключения

Что делает ячейка:
- оценивает семантическую согласованность введения и заключения с основной частью,
- сравнивает эти значения с baseline `main-main`.


In [6]:
def upper_triangle_values(M: np.ndarray) -> np.ndarray:
    if M.shape[0] < 2:
        return np.array([], dtype=np.float32)
    iu = np.triu_indices_from(M, k=1)
    return M[iu].astype(np.float32)


intro_to_main_mean = float("nan")
conc_to_main_mean = float("nan")
main_main_mean = float("nan")

if main_idx.size:
    main_block = S[np.ix_(main_idx, main_idx)]
    mm_vals = upper_triangle_values(main_block)
    if mm_vals.size:
        main_main_mean = float(mm_vals.mean())

if intro_i is not None and main_idx.size:
    intro_to_main_mean = float(S[intro_i, main_idx].mean())

if conc_i is not None and main_idx.size:
    conc_to_main_mean = float(S[conc_i, main_idx].mean())

intro_delta = float(intro_to_main_mean - main_main_mean) if np.isfinite(intro_to_main_mean) and np.isfinite(main_main_mean) else float("nan")
conc_delta = float(conc_to_main_mean - main_main_mean) if np.isfinite(conc_to_main_mean) and np.isfinite(main_main_mean) else float("nan")

print("intro->main mean:", round(intro_to_main_mean, 4) if np.isfinite(intro_to_main_mean) else "nan")
print("conc->main mean:", round(conc_to_main_mean, 4) if np.isfinite(conc_to_main_mean) else "nan")
print("main-main mean:", round(main_main_mean, 4) if np.isfinite(main_main_mean) else "nan")


intro->main mean: 0.6599
conc->main mean: 0.6447
main-main mean: 0.635


## 7) Избыточность и связность разделов

Что делает ячейка:
- считает избыточность в основной части (`redund_90`),
- оценивает связность внутри разделов и между разделами.


In [7]:
redund_90 = float("nan")
if main_idx.size >= 2:
    M = S[np.ix_(main_idx, main_idx)]
    main_vals = upper_triangle_values(M)
    redund_90 = float(np.mean(main_vals > REDUNDANCY_THRESHOLD)) if main_vals.size else float("nan")

within_section_values = []
for _, idxs in parent_to_indices.items():
    idxs = np.array(idxs, dtype=np.int64)
    if idxs.size < 2:
        continue
    block = S[np.ix_(idxs, idxs)]
    vals = upper_triangle_values(block)
    if vals.size:
        within_section_values.append(float(vals.mean()))

within_section_mean = float(np.mean(within_section_values)) if within_section_values else float("nan")

cross_section_values = []
parent_keys = list(parent_to_indices.keys())
for i in range(len(parent_keys)):
    for j in range(i + 1, len(parent_keys)):
        a = np.array(parent_to_indices[parent_keys[i]], dtype=np.int64)
        b = np.array(parent_to_indices[parent_keys[j]], dtype=np.int64)
        if a.size == 0 or b.size == 0:
            continue
        block = S[np.ix_(a, b)]
        cross_section_values.append(float(block.mean()))

cross_section_mean = float(np.mean(cross_section_values)) if cross_section_values else float("nan")

print("redund_90:", round(redund_90, 4) if np.isfinite(redund_90) else "nan")
print("within section mean:", round(within_section_mean, 4) if np.isfinite(within_section_mean) else "nan")
print("cross section mean:", round(cross_section_mean, 4) if np.isfinite(cross_section_mean) else "nan")


redund_90: 0.0
within section mean: nan
cross section mean: 0.6514


## 8) Правила проблем и итоговый рискскор

Что делает ячейка:
- формирует флаги проблем с понятными описаниями,
- считает итоговый балл риска для быстрой навигации.

Важно:
- итоговый балл риска вспомогателен и не заменяет интерпретацию каждой метрики.


In [8]:
problems = []

# Правило 1: слабая локальная связность.
if np.isfinite(adj_mean) and adj_mean < 0.65:
    problems.append({
        "type": "low_local_coherence",
        "severity": "high",
        "metric": "adj_mean",
        "value": float(adj_mean),
        "threshold": 0.65,
        "message": "Average adjacent similarity is low; transitions may be weak.",
    })

# Правило 2: много слабых переходов по Z-score.
if adj.size and (len(weak_transition_indices) / adj.size) > 0.20:
    problems.append({
        "type": "many_weak_transitions",
        "severity": "high",
        "metric": "weak_transition_share",
        "value": float(len(weak_transition_indices) / adj.size),
        "threshold": 0.20,
        "message": "High share of transition outliers; document may have logical jumps.",
    })

# Правило 3: согласованность введения.
if np.isfinite(intro_delta) and intro_delta < -0.05:
    problems.append({
        "type": "intro_main_misalignment",
        "severity": "medium",
        "metric": "intro_delta",
        "value": float(intro_delta),
        "threshold": -0.05,
        "message": "Introduction is semantically weakly aligned with main body.",
    })

# Правило 4: согласованность заключения.
if np.isfinite(conc_delta) and conc_delta < -0.05:
    problems.append({
        "type": "conclusion_main_misalignment",
        "severity": "medium",
        "metric": "conc_delta",
        "value": float(conc_delta),
        "threshold": -0.05,
        "message": "Conclusion is semantically weakly aligned with main body.",
    })

# Правило 5: избыточность.
if np.isfinite(redund_90) and redund_90 > 0.25:
    problems.append({
        "type": "high_redundancy",
        "severity": "medium",
        "metric": "redund_90",
        "value": float(redund_90),
        "threshold": 0.25,
        "message": "Main part contains many near-duplicate semantic pairs.",
    })

# Итоговый балл риска для быстрой оценки (примерно 0..100).
risk_score = 0.0
if np.isfinite(adj_mean):
    risk_score += max(0.0, (0.70 - adj_mean)) * 120.0
if adj.size:
    risk_score += (len(weak_transition_indices) / adj.size) * 35.0
if np.isfinite(intro_delta):
    risk_score += max(0.0, -intro_delta) * 80.0
if np.isfinite(conc_delta):
    risk_score += max(0.0, -conc_delta) * 80.0
if np.isfinite(redund_90):
    risk_score += max(0.0, redund_90 - 0.10) * 70.0

risk_score = float(min(100.0, max(0.0, risk_score)))

print("problem flags:", len(problems))
print("risk_score:", round(risk_score, 2))


problem flags: 0
risk_score: 10.68


## 9) Формирование и сохранение отчета

Что делает ячейка:
- собирает структурированный итоговый отчет,
- сохраняет отчет в `out/document_metrics_report.json`.


In [9]:
report = {
    "model": "FRIDA",
    "n_segments": int(n_segments),
    "local_coherence": {
        "adj_mean": float(adj_mean),
        "adj_std": float(adj_std),
        "weak_transition_count": int(len(weak_transition_indices)),
        "weak_transition_share": float(len(weak_transition_indices) / adj.size) if adj.size else float("nan"),
    },
    "structure_consistency": {
        "intro_to_main_mean": float(intro_to_main_mean) if np.isfinite(intro_to_main_mean) else None,
        "conclusion_to_main_mean": float(conc_to_main_mean) if np.isfinite(conc_to_main_mean) else None,
        "main_main_mean": float(main_main_mean) if np.isfinite(main_main_mean) else None,
        "intro_delta": float(intro_delta) if np.isfinite(intro_delta) else None,
        "conclusion_delta": float(conc_delta) if np.isfinite(conc_delta) else None,
    },
    "redundancy": {
        "threshold": float(REDUNDANCY_THRESHOLD),
        "redund_90": float(redund_90) if np.isfinite(redund_90) else None,
    },
    "section_coherence": {
        "within_section_mean": float(within_section_mean) if np.isfinite(within_section_mean) else None,
        "cross_section_mean": float(cross_section_mean) if np.isfinite(cross_section_mean) else None,
    },
    "weakest_transitions_top_n": weakest_transitions,
    "problem_flags": problems,
    "risk_score": float(risk_score),
}

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", REPORT_PATH)


saved: out\document_metrics_report.json


## 10) Краткая сводка для ручной проверки

Эта ячейка выводит краткую итоговую информацию для быстрого просмотра.


In [10]:
print("МОДЕЛЬ:", report["model"])
print("СЕГМЕНТОВ:", report["n_segments"])
print("БАЛЛ РИСКА:", round(report["risk_score"], 2))

print()
print("Флаги проблем:")
if report["problem_flags"]:
    for p in report["problem_flags"]:
        print("-", p["type"], "|", p["severity"], "|", p["metric"], "=", round(float(p["value"]), 4))
else:
    print("- none")

print()
print("Самые слабые переходы:")
if report["weakest_transitions_top_n"]:
    for t in report["weakest_transitions_top_n"]:
        print(
            f"- [{t['from_index']} -> {t['to_index']}] ",
            f"sim={t['adj_similarity']:.4f}, z={t['adj_z']:.3f}",
        )
        print("  from:", t["from_path"])
        print("  to  :", t["to_path"])
else:
    print("- none")


МОДЕЛЬ: FRIDA
СЕГМЕНТОВ: 6
БАЛЛ РИСКА: 10.68

Флаги проблем:
- none

Самые слабые переходы:
- [4 -> 5]  sim=0.4960, z=-1.491
  from: Примеры работы программы
  to  : Заключение
- [3 -> 4]  sim=0.6206, z=-0.419
  from: Разработка программного обеспечения
  to  : Примеры работы программы
- [2 -> 3]  sim=0.6520, z=-0.149
  from: Выбор средств реализации
  to  : Разработка программного обеспечения
- [1 -> 2]  sim=0.7334, z=0.551
  from: Постановка задачи
  to  : Выбор средств реализации
- [0 -> 1]  sim=0.8445, z=1.508
  from: Введение
  to  : Постановка задачи
